# IOAI — 2025 Summer Online University Admission (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2025/nyari-online/adatok >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2025/nyari-online/adatok/*'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 대학 입시 예측 (University Admission / Egyetemi Felvételi) — 모범답안

HAIO 2025 여름 온라인 예선 (ML). 학생 특징으로 **입학 확률**(`Felvételi Eredmény`, 0/1)을 예측. 점수 = **ROC-AUC**.
공식 `test.csv` 는 라벨 비공개(Kaggle)라 `train.csv` 의 **held-out 분할**(`index % 5 == 0`, 1377명)로 채점.
제출 `submission.csv`(`ID, Felvételi Eredmény` 확률).

**핵심 발견 — 입시 결정은 사실상 *선형*이다.** 여러 모델을 비교하면:
- 베이스라인 HistGradientBoosting(트리) → ROC-AUC ≈ **0.933**
- **피처 엔지니어링 + 표준화 + 로지스틱 회귀** → ROC-AUC ≈ **0.948** ← 트리를 확실히 앞섬

성적·과목·부모학력·추천서 등이 입학 로그오즈에 **거의 가산적으로** 작용하므로, 잘 스케일된 선형 모델이
트리보다 잘 일반화한다(트리는 0.934 근처에서 정체). 트리 앙상블 블렌딩을 더해도 향상은 노이즈 수준(≈0.948) →
**단순·정직하게 선형 모델을 채택**한다.

**두 가지 포인트**:
1. **`-1`(과목 미응시) 센티널을 그대로 둔다** — 이 값을 NaN 으로 지우면 오히려 나빠진다(0.929). `-1` 자체가
   "그 과목을 안 봤다"는 선형적으로 유용한 신호다. (단, 파생 통계는 `-1→NaN` 으로 계산.)
2. **Vármegye(주)는 스무딩 타깃인코딩** — *학습 분할에서만* 통계를 내 누수를 막는다.


In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("train.csv").reset_index(drop=True)
T = "Felvételi Eredmény"
is_val = (df.index % 5 == 0)                 # 공식 채점과 동일한 held-out 분할
tr, va = df[~is_val].copy(), df[is_val].copy()
ytr, yva = tr[T].to_numpy(), va[T].to_numpy()
print("train", tr.shape, "val", va.shape, "| 합격률", round(ytr.mean(), 3))

SUBJ  = ["Történelem","Matematika","Magyar Nyelv és Irodalom","Informatika","Biológia","Fizika","Angol","Német"]
EMELT = [s+"_emelt" for s in ["Informatika","Biológia","Fizika","Angol","Német","Matematika","Történelem","Magyar Nyelv és Irodalom"]]
GRADE = ["Osztályzat_9","Osztályzat_10","Osztályzat_11","Osztályzat_12"]


In [ ]:
# 스무딩 타깃인코딩(학습 분할에서만 → 누수 없음)
def county_encoding(tr_df, y, alpha=20):
    g = pd.DataFrame({"c": tr_df["Vármegye"].values, "y": y})
    stat = g.groupby("c")["y"].agg(["mean", "count"]); glob = y.mean()
    return ((stat["mean"]*stat["count"] + glob*alpha) / (stat["count"] + alpha)).to_dict(), glob
cmap, glob = county_encoding(tr, ytr)

def fe(dframe):                              # -1 센티널은 유지, 파생통계는 -1→NaN 으로 계산
    d = dframe.copy(); sub = d[SUBJ].replace(-1, np.nan)
    d["subj_mean"] = sub.mean(1); d["subj_max"] = sub.max(1); d["subj_min"] = sub.min(1); d["subj_std"] = sub.std(1)
    d["n_subj_taken"] = (d[SUBJ] != -1).sum(1); d["n_emelt"] = (d[EMELT] == 1).sum(1)
    d["grade_mean"] = d[GRADE].mean(1); d["grade_trend"] = d["Osztályzat_12"] - d["Osztályzat_9"]
    d["grade_last2"] = d[["Osztályzat_11","Osztályzat_12"]].mean(1)
    d["county_te"] = d["Vármegye"].map(cmap).fillna(glob)
    return d.drop(columns=["Vármegye"])

Ftr, Fva = fe(tr.drop(columns=["ID", T])), fe(va.drop(columns=["ID", T]))
med = Ftr.median()                           # 스케일러 fit 은 학습 분할 통계로만
scaler = StandardScaler().fit(Ftr.fillna(med).values)
Str, Sva = scaler.transform(Ftr.fillna(med).values), scaler.transform(Fva.fillna(med).values)
print("피처 수:", Ftr.shape[1])


In [ ]:
# 로지스틱 회귀 → held-out 예측 → submission.csv
clf = LogisticRegression(C=0.3, max_iter=3000).fit(Str, ytr)
proba = clf.predict_proba(Sva)[:, 1]
print(f"Held-out ROC-AUC: {roc_auc_score(yva, proba):.4f}")
pd.DataFrame({"ID": va["ID"].to_numpy(), T: proba}).to_csv("submission.csv", index=False)
print("wrote submission.csv", len(va))


### 정리
- **선형 모델이 이긴다**: 피처 엔지니어링 + 표준화 + LogisticRegression → ROC-AUC ≈ **0.948** (트리 베이스라인 ≈0.933).
  입시 점수화가 특징들의 (거의) 가산적 함수라 잘 스케일된 선형 결정경계가 트리보다 잘 일반화한다.
- **`-1`(미응시) 센티널 유지**가 핵심(지우면 0.929로 하락). Vármegye 는 *학습 분할에서만* 스무딩 타깃인코딩(누수 방지).
- **더 시도**: HistGBM 과의 랭크 블렌딩(≈0.948, 향상 미미), 상호작용항, 과목 결측 패턴 임베딩. 단순·정직성 우선이라
  본문은 선형 단일모델을 채택했다.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)